# 02 - Raw BLD versus Fixed and Adaptive CCI

This independent smile-only notebook checks out a pinned repository revision and runs one standalone, unbuffered command. It compares raw BLD, fixed-equal CCI, and adaptive-feedback CCI using mouth plus both lips.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time, torch
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/lokissdo/cci-diff.git'
GIT_REF = 'main'
IS_KAGGLE = Path('/kaggle/working').is_dir()
PROJECT_ROOT = Path('/kaggle/working/cci-diff') if IS_KAGGLE else Path.cwd().resolve()
if not IS_KAGGLE and not (PROJECT_ROOT / 'scripts' / 'run_kaggle_smile.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_ROOT = Path('/kaggle/working/cci_fixed_vs_adaptive') if IS_KAGGLE else PROJECT_ROOT / 'outputs' / 'cci_fixed_vs_adaptive'
MODEL_PATH = 'sd2-community/stable-diffusion-2-1'
SAMPLE_COUNT = 300
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
MAX_WORKERS = torch.cuda.device_count() if DEVICE == 'cuda' else 1
ASSET_ROOT = Path('/kaggle/input/datasets/a210462khihng/cci-assets') if IS_KAGGLE else PROJECT_ROOT / 'models'
CELEBA_ROOT = Path('/kaggle/input/datasets/ipythonx/celebamaskhq/CelebAMask-HQ') if IS_KAGGLE else PROJECT_ROOT / 'data' / 'CelebAMask-HQ'
CLASSIFIER_PATH = ASSET_ROOT / 'resnet50_multilabel_model.pth'
IDENTITY_MODEL_PATH = ASSET_ROOT / 'facenet_vggface2.ts'
IMAGE_ROOT = CELEBA_ROOT / 'CelebA-HQ-img'
MASK_ROOT = CELEBA_ROOT / 'CelebAMask-HQ-mask-anno'
NUM_INFERENCE_STEPS = 35
SEED = 42

In [ ]:
def bootstrap(label, command, cwd=None):
    print(f'[{time.strftime("%H:%M:%S")}] START {label}', flush=True)
    print('+ ' + ' '.join(str(value) for value in command), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)
    print(f'[{time.strftime("%H:%M:%S")}] DONE  {label}', flush=True)

if IS_KAGGLE:
    if not (PROJECT_ROOT / '.git').is_dir():
        bootstrap('bootstrap: clone repository', ['git', 'clone', '--no-checkout', REPO_URL, PROJECT_ROOT])
    bootstrap('bootstrap: fetch pinned revision', ['git', 'fetch', '--depth', '1', 'origin', GIT_REF], cwd=PROJECT_ROOT)
    bootstrap('bootstrap: checkout pinned revision', ['git', 'checkout', '--force', '--detach', 'FETCH_HEAD'], cwd=PROJECT_ROOT)

command = [
    sys.executable, '-u', PROJECT_ROOT / 'scripts' / 'run_kaggle_smile.py',
    '--mode', 'evaluation', '--sample_count', str(SAMPLE_COUNT),
    '--model_path', MODEL_PATH, '--device', DEVICE, '--max_workers', str(MAX_WORKERS),
    '--classifier_path', CLASSIFIER_PATH, '--identity_model_path', IDENTITY_MODEL_PATH,
    '--image_root', IMAGE_ROOT, '--mask_root', MASK_ROOT,
    '--num_inference_steps', str(NUM_INFERENCE_STEPS), '--seed', str(SEED),
    '--output_dir', OUTPUT_ROOT,
]
print(f'[{time.strftime("%H:%M:%S")}] START standalone smile evaluation', flush=True)
print('+ ' + ' '.join(str(value) for value in command), flush=True)
subprocess.run(command, check=True)
print(f'[{time.strftime("%H:%M:%S")}] DONE  standalone smile evaluation', flush=True)

In [ ]:
import pandas as pd
results = pd.read_csv(OUTPUT_ROOT / 'pilot_results.csv')
results['controller_mode'] = results['variant'].map({'A0': 'raw_bld', 'A2': 'fixed_equal', 'A3': 'feedback'})
display(results.groupby(['feature', 'controller_mode']).agg(
    count=('target_pass', 'size'), generation_classifier_fr=('target_pass', 'mean'),
    desired_probability=('desired_probability', 'mean'), identity_cosine=('identity_cosine', 'mean'),
    non_target_drift=('non_target_drift', 'mean'), changed_fraction_5=('changed_fraction_5', 'mean'),
    outside_semantic_fraction_5=('outside_semantic_fraction_5', 'mean'),
    residual_tv=('residual_tv', 'mean'), runtime_seconds=('runtime_seconds', 'mean'),
).reset_index())
print('Source/output comparisons:', OUTPUT_ROOT / 'comparisons', flush=True)